# Семинар №8. Программный доступ к БД из Python

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Полные решения собраны в разделе «Решения» в самом конце. Старайтесь сначала решить сами.
4. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно (мы переиспользуем одни и те же таблицы).
5. Сброс: `Kernel → Restart & Run All`.

## Окружение

- PostgreSQL 16 слушает `localhost:5432` **внутри контейнера**.
- Учётные данные БД: пользователь `postgres`, **пароль не требуется** (trust auth для локальных подключений), база `demo`.
- Jupyter-пароль: `student`.

## План

1. `psycopg` (DB-API): соединения, курсоры, параметризация.
2. Транзакции и `with conn:`.
3. SQL-инъекции и почему f-string в SQL — зло.
4. Батчевая вставка (`executemany`).
5. SQLAlchemy Core (`text()`).
6. Pandas: `read_sql` и `chunksize`.
7. ORM: модели, M:N, каскады.
8. Проблема N+1 и `selectinload`.


In [4]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, sqlalchemy, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("sqlalchemy  :", sqlalchemy.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

# Параметры подключения (внутри контейнера localhost = этот же контейнер).
# PostgreSQL настроен с trust-аутентификацией для локальных подключений,
# поэтому пароль НЕ требуется и НЕ передаётся.
DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "demo",
    "user":   "postgres",
}

PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} "
    f"user={DB['user']}"
)
SQLALCHEMY_URL = (
    f"postgresql+psycopg://{DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
)

print("\npsycopg DSN      :", PSYCOPG_CONN)
print("SQLAlchemy URL   :", SQLALCHEMY_URL)


Python: 3.11.17
psycopg     : 3.3.6
sqlalchemy  : 2.1.6
pandas      : 3.1.0
numpy       : 2.4.6

psycopg DSN      : host=localhost port=5432 dbname=demo user=postgres
SQLAlchemy URL   : postgresql+psycopg://postgres@localhost:5432/demo


## 1. Основы `psycopg` (DB-API)

**DB-API** (PEP 249) — стандарт Python для работы с реляционными БД: `connect()`, `cursor()`, `execute()`, `fetch*()`. У каждого драйвера свой диалект, но интерфейс общий.

Пакет `psycopg` (v3) — современный драйвер PostgreSQL. Что важно знать:

- **Соединение** открывается через `psycopg.connect(conn_string)`; его **обязательно** закрывать. Лучший способ — менеджер контекста `with psycopg.connect(...) as conn`.
- **Курсор** — объект, через который выполняются запросы; тоже умеет `with`.
- `cur.execute(sql, params)` — два аргумента: SQL-шаблон и кортеж/список параметров. Драйвер сам корректно экранирует и передаёт значения по бинарному протоколу. **Никогда не склеивайте SQL строками**.
- `cur.fetchone()` / `cur.fetchall()` / итерация по `cur` — способы получить результат.
- `psycopg.connect(..., row_factory=dict_row)` — строки как словари вместо кортежей.
- `psycopg.connect(...)` при выходе из `with` вызывает `close()`. Обратите внимание: `commit/rollback` на этом уровне **не** делается — это уже часть транзакций (см. тему 2).
- В реальных веб-приложениях напрямую `connect` почти не используют — берут пул (`psycopg_pool.ConnectionPool` или `QueuePool` в SQLAlchemy), потому что открытие нового TCP-соединения и форк бэкенд-процесса PG — дорого.


In [5]:
import psycopg
from psycopg.rows import dict_row

# --- 1. Версия сервера и типичный SELECT ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print("PG version:", cur.fetchone()[0].split(" on ")[0])

# --- 2. DDL + DML + чтение в одной транзакции ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS demo_items")
        cur.execute("CREATE TABLE demo_items (id INT PRIMARY KEY, name TEXT, qty INT)")
        cur.execute(
            "INSERT INTO demo_items (id, name, qty) VALUES (%s, %s, %s), (%s, %s, %s)",
            (1, "apple", 10, 2, "banana", 20),
        )
    conn.commit()

    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items ORDER BY id")
        print("\nrows (tuples):")
        for row in cur.fetchall():
            print(" ", row)

# --- 3. То же самое, но со словарями ---
with psycopg.connect(PSYCOPG_CONN, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items ORDER BY id")
        print("\nrows (dict_row):")
        for row in cur.fetchall():
            print(" ", row)


PG version: PostgreSQL 15.19 (Debian 15.19-0+deb12u1)

rows (tuples):
  (1, 'apple', 10)
  (2, 'banana', 20)

rows (dict_row):
  {'id': 1, 'name': 'apple', 'qty': 10}
  {'id': 2, 'name': 'banana', 'qty': 20}


### Упражнение 1. Параметризованный поиск

**Задание.** В таблице `demo_items` найдите запись, где `name` совпадает с `target_name`, и выведите `id, name, qty`.

**Шаги.**
1. Откройте соединение `psycopg.connect(PSYCOPG_CONN)` и курсор.
2. Выполните `SELECT id, name, qty FROM demo_items WHERE name = %s` с параметром `(target_name,)`.
3. Напечатайте результат.
4. Дополнительно: поменяйте `target_name` на `"apple' OR '1'='1"` и убедитесь, что запрос вернул 0 строк — инъекция не сработала.

**Критерий успеха.** При `target_name = 'apple'` печатается `(1, 'apple', 10)`; при «инъекционном» вводе — пусто.


In [5]:
# УПРАЖНЕНИЕ 1
# psycopg (Основы): Это «переводчик» между Python и PostgreSQL. С его помощью мы отправляем SQL-запросы из кода.
# Цель: Научиться безопасно передавать переменные из Python в SQL-запрос, чтобы хакеры не могли взломать базу.

target_name = "apple"  # затем попробуйте: "apple' OR '1'='1"

import psycopg

# 1. Открываем соединение с базой
with psycopg.connect(PSYCOPG_CONN) as conn:
    # 2. Создаем курсор (объект, через который отправляем команды)
    with conn.cursor() as cur:
        # 3. Выполняем SQL запрос
        # execute - от анг. выполнять
        cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", (target_name,))
        # 4. Получаем все строки, которые вернула база
        for row in cur.fetchall():
             print(row)
            


## 2. Транзакции и `with conn:`

- В PostgreSQL транзакция начинается с первого SQL-запроса.
- В psycopg3 у соединения есть метод `conn.transaction()`, но проще — **менеджер контекста `with conn:`**. Он делает `commit()` при успешном выходе и `rollback()` при любом исключении.
- **DDL транзакционен** (в отличие, например, от MySQL). Если вы упадёте после `CREATE TABLE` — таблица откатится.
- **Долгие транзакции — зло.** Пока транзакция открыта, PG не может почистить старые версии строк через `autovacuum` (MVCC, семинар 5). Отсюда bloat. Правило: транзакция должна длиться миллисекунды, а не минуты. Не держите транзакцию, пока ждёте ответа от медленного API.


In [6]:
# Транзакции (with conn:): Защита от ошибок. Если в коде что-то упало на середине, 
# база откатит все изменения назад (как кнопка «Отменить» в Word).

from psycopg.errors import UniqueViolation

# Готовим таблицу ВНЕ проблемной транзакции
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS tx_demo")
        cur.execute("CREATE TABLE tx_demo (id INT PRIMARY KEY, name TEXT)")
    conn.commit()

try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # Дубликат PK -> вторая строка упадёт
            cur.executemany("INSERT INTO tx_demo VALUES (%s, %s)", [(1, "a"), (1, "b")])
        conn.commit()
except UniqueViolation as e:
    print("Поймали UniqueViolation:", str(e).splitlines()[0])
    # with conn: автоматически сделает rollback

# Проверяем: в таблице 0 строк (вставка откатилась целиком)
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM tx_demo")
        print("rows after rollback:", cur.fetchone()[0])


Поймали UniqueViolation: duplicate key value violates unique constraint "tx_demo_pkey"
rows after rollback: 0


### Упражнение 2. Атомарность при ошибке

**Задание.** Смоделируйте ситуацию, где часть строк вставляется успешно, а затем происходит ошибка, и убедитесь, что откатилось всё.

**Шаги.**
1. В отдельной транзакции создайте таблицу `ex2 (id INT PRIMARY KEY, name TEXT)` — без дубликатов.
2. В следующей транзакции выполните `executemany` со списком `[(1,'a'), (2,'b'), (1,'dup')]`.
3. Оберните в `try/except UniqueViolation`.
4. Проверьте `SELECT count(*) FROM ex2` — должно быть 0 (откатились даже успешные вставки).

**Критерий успеха.** Счётчик 0, в консоли — сообщение об ошибке.


In [22]:
# УПРАЖНЕНИЕ 2
# Транзакции: «всё или ничего». Если в пачке одна строка битая, откатывается ВСЯ пачка
# (как кнопка «Отменить» в Word, только для целой группы действий).

from psycopg.errors import UniqueViolation

# --- Шаг 1. Создаём таблицу в ОТДЕЛЬНОЙ транзакции и фиксируем её ---
# Если бы CREATE TABLE был в той же транзакции, что и ошибочная вставка,
# при откате исчезла бы и таблица (DDL в PostgreSQL транзакционен),
# и SELECT count(*) упал бы с ошибкой вместо того, чтобы вернуть 0.
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS ex2")
        cur.execute("CREATE TABLE ex2 (id INT PRIMARY KEY, name TEXT)")
    conn.commit()   # таблица зафиксирована, откат её уже не заденет

# --- Шаг 2-3. Все три вставки — в ОДНОЙ транзакции; try стоит СНАРУЖИ with ---
try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # (1,'a') и (2,'b') вставятся, а (1,'dup') нарушит PRIMARY KEY -> UniqueViolation
            cur.executemany(
                "INSERT INTO ex2 VALUES (%s, %s)",
                [(1, "a"), (2, "b"), (1, "dup")],
            )
        conn.commit()   # сюда выполнение не дойдёт: исключение вылетит раньше
except UniqueViolation as e:
    # Исключение «пролетело» через блок with -> psycopg сам сделал ROLLBACK и закрыл соединение
    print("Поймали UniqueViolation:", str(e).splitlines()[0])

# --- Шаг 4. Проверяем НОВЫМ соединением ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM ex2")
        print("Строк в таблице:", cur.fetchone()[0])   # 0 — откатились даже «хорошие» (1,'a') и (2,'b')


# --- БОНУС: если часть данных ЗАФИКСИРОВАТЬ до ошибки, она останется ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO ex2 VALUES (%s, %s)", [(1, "a"), (2, "b")])
    conn.commit()                                   # транзакция №1 зафиксирована

try:
    with psycopg.connect(PSYCOPG_CONN) as conn:     # транзакция №2 — отдельная
        with conn.cursor() as cur:
            cur.execute("INSERT INTO ex2 VALUES (%s, %s)", (1, "dup"))   # дубль -> ошибка
except UniqueViolation:
    print("транзакция №2 откатилась")

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM ex2")
        print("Строк после бонуса:", cur.fetchone()[0])   # 2 — откатывается только транзакция с ошибкой

Поймали UniqueViolation: duplicate key value violates unique constraint "ex2_pkey"
Строк в таблице: 0
транзакция №2 откатилась
Строк после бонуса: 2


## 3. SQL-инъекции и параметризация

**SQL-инъекция** — это когда пользовательский ввод попадает в строку SQL и интерпретируется как код.

- ❌ **Уязвимо:** `cur.execute(f"... WHERE name = '{name}'")`. Если `name = "' OR '1'='1"`, запрос вернёт все строки; если `name = "'; DROP TABLE ...; --"` — таблица исчезнет.
- ✅ **Безопасно:** `cur.execute("... WHERE name = %s", (name,))`. Данные и SQL идут раздельно, значения передаются по бинарному протоколу и **не могут быть интерпретированы как код**.
- То же в SQLAlchemy Core: `text("... WHERE name = :name")` + `{"name": ...}`. **Никогда** не подставляйте значения внутрь `text()` через f-строку.


In [9]:
def unsafe_search(name):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # ❌ f-string — SQL-инъекция
            cur.execute(f"SELECT id, name, qty FROM demo_items WHERE name = '{name}'")
            return cur.fetchall()

def safe_search(name):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # ✅ параметризация
            cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", (name,))
            return cur.fetchall()

malicious = "' OR '1'='1' --" # пример хакерской атаки
print("unsafe :", unsafe_search(malicious))
print("safe   :", safe_search(malicious))


unsafe : [(1, 'apple', 10), (2, 'banana', 20)]
safe   : []


### Упражнение 3. Почините уязвимый запрос

**Задание.** Дана функция `broken(code)` — она ищет пилота по коду через f-string. Перепишите её безопасно.

**Шаги.**
1. Скопируйте функцию, замените f-string на `%s` + параметр.
2. Проверьте: при `code = "' OR '1'='1"` результат должен быть пустым.
3. Дополнительно: добавьте `LIMIT 5` в SQL.

**Критерий успеха.** `broken("' OR '1'='1")` (старая версия) вернёт всё, `fixed("' OR '1'='1")` — пусто.


In [13]:
# УПРАЖНЕНИЕ 3
# SQL-инъекции: Главная уязвимость хакеров. 
# Если ты подставляешь данные пользователя прямо в строку SQL, хакер может удалить базу. Решение — параметризация.

def broken(code):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, name FROM demo_items WHERE name = '{code}'")
            return cur.fetchall()

def fixed(code):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, name FROM demo_items WHERE name = %s LIMIT 5", (code,)) # ПРОВЕРЯЕТ РАВНО ЛИ ИМЯ ("' OR '1'='1' --" # пример хакерской атаки)
            return cur.fetchall()

# TODO: напишите безопасную версию fixed(code)
# TODO: проверьте на "' OR '1'='1"

code = "' OR '1'='1' --" # пример хакерской атаки
print("unsafe :", broken(code))
print("safe   :", fixed(code))


unsafe : [(1, 'apple'), (2, 'banana')]
safe   : []


## 4. Батчевая вставка: `executemany`

- Цикл `for row in rows: cur.execute("INSERT ...", row)` — это **N сетевых round-trip'ов** и N записей в WAL. Медленно.
- `cur.executemany(sql, list_of_tuples)` — один вызов, драйвер группирует строки в пакеты.
- В `psycopg3` (≥3.1) `executemany` дополнительно объединяет несколько `INSERT` в один `pipeline`-пакет.
- Для по-настоящему больших объёмов (миллионы строк) есть отдельный, ещё более быстрый протокол — `COPY` (`cur.copy()`), но `executemany` — уже огромный шаг вперёд по сравнению с циклом.


In [6]:
# Импортируем модуль time для замера скорости (секундомер)
# и модуль random для генерации случайных чисел
import psycopg
import time, random

# Задаем количество строк, которые будем вставлять
N = 3000 

# Создаем список (массив) из N кортежей (tuples).
# Внутри цикла генерируем: (порядковый_номер, строка_с_именем, случайное_число_от_1_до_100)
# Пример одного элемента: (1, "item_1", 42)
rows = [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, N + 1)]

# ==========================================
# ПОДГОТОВКА БАЗЫ ДАННЫХ
# ==========================================
# Открываем соединение с базой (with автоматически закроет его в конце)
with psycopg.connect(PSYCOPG_CONN) as conn:
    # Открываем курсор (объект для отправки SQL-команд)
    with conn.cursor() as cur:
        # Удаляем старую таблицу, если она осталась от прошлых запусков
        cur.execute("DROP TABLE IF EXISTS bench")
        # Создаем новую чистую таблицу с тремя колонками
        cur.execute("CREATE TABLE bench (id INT PRIMARY KEY, name TEXT, val INT)")
    # Обязательно сохраняем изменения (создание таблицы) в базе
    conn.commit()

# ==========================================
# МЕТОД 1: ВСТАВКА В ЦИКЛЕ (МЕДЛЕННО)
# ==========================================
# Включаем секундомер
t0 = time.perf_counter()

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        # Проходимся по каждой строке из нашего списка
        for r in rows:
            # И делаем ОТДЕЛЬНЫЙ запрос INSERT для каждой строки!
            # Это создает N сетевых запросов к базе.
            cur.execute("INSERT INTO bench VALUES (%s, %s, %s)", r)
    # Сохраняем все вставки
    conn.commit()

# Выключаем секундомер и вычисляем затраченное время (в секундах)
t_loop = time.perf_counter() - t0
print(f"цикл         : {t_loop:.2f}s")

# ==========================================
# ПОДГОТОВКА КО ВТОРОМУ ТЕСТУ
# ==========================================
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        # TRUNCATE быстро очищает таблицу от данных (но оставляет саму структуру).
        # Это быстрее, чем DELETE.
        cur.execute("TRUNCATE bench")
    conn.commit()

# ==========================================
# МЕТОД 2: ВСТАВКА ПАЧКОЙ executemany (БЫСТРО)
# ==========================================
t0 = time.perf_counter()

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        # Магия! Вместо N запросов, executemany собирает все данные
        # в один большой пакет и отправляет в базу ЗА ОДИН РАЗ.
        # Это экономит время на сетевом пинг-понге.
        cur.executemany("INSERT INTO bench VALUES (%s, %s, %s)", rows)
    conn.commit()

t_batch = time.perf_counter() - t0
print(f"executemany  : {t_batch:.2f}s")

# ==========================================
# ВЫВОД РЕЗУЛЬТАТА
# ==========================================
# Делим время цикла на время батча, чтобы узнать ускорение.
# max(t_batch, 1e-9) защищает от ошибки деления на ноль (если батч отработает моментально).
print(f"ускорение    : x{t_loop / max(t_batch, 1e-9):.1f}")

цикл         : 0.27s
executemany  : 0.03s
ускорение    : x10.2


### Упражнение 4. Батч на 10 000 строк

**Задание.** Сравните цикл и `executemany` на 10 000 строк и найдите порог, на котором `executemany` стабильно быстрее.

**Шаги.**
1. Сгенерируйте 10 000 кортежей `(id, name, val)`.
2. Замерьте цикл, замерьте `executemany`.
3. Напечатайте `speedup = t_loop / t_batch`.
4. **Вопрос на подумать:** почему `executemany` не «в N раз быстрее» линейно?

**Критерий успеха.** `speedup > 5`. В комментарии — 1–2 строки объяснения.


In [23]:
# УПРАЖНЕНИЕ 4
# Цель: увидеть, что executemany (пачка) быстрее цикла из execute, и найти порог выигрыша.

import time, random
import psycopg

# --- Таблица для замеров ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench10k")
        cur.execute("CREATE TABLE bench10k (id INT PRIMARY KEY, name TEXT, val INT)")
    conn.commit()

def make_rows(n):
    """n кортежей (id, name, val)."""
    return [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, n + 1)]

def truncate():
    """Очищаем таблицу ВНЕ замера (иначе будет конфликт id и лишнее время)."""
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute("TRUNCATE bench10k")
        conn.commit()

def time_loop(rows):
    """Цикл: один execute на строку -> N обращений к серверу."""
    truncate()
    t0 = time.perf_counter()                       # включили секундомер
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            for r in rows:
                cur.execute("INSERT INTO bench10k VALUES (%s, %s, %s)", r)
        conn.commit()
    return time.perf_counter() - t0                # сколько секунд прошло

def time_batch(rows):
    """Пачка: один вызов executemany на все строки."""
    truncate()
    t0 = time.perf_counter()
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.executemany("INSERT INTO bench10k VALUES (%s, %s, %s)", rows)
        conn.commit()
    return time.perf_counter() - t0

# --- Ищем порог: с какого размера батч заметно выигрывает ---
print(f"{'N':>7} | {'цикл, c':>8} | {'batch, c':>8} | ускорение")
for n in (10, 100, 1000, 10_000):
    rows = make_rows(n)
    t_loop, t_batch = time_loop(rows), time_batch(rows)
    print(f"{n:>7} | {t_loop:>8.3f} | {t_batch:>8.3f} | x{t_loop / max(t_batch, 1e-9):.1f}")

# --- Основной результат: 10 000 строк ---
rows = make_rows(10_000)
t_loop, t_batch = time_loop(rows), time_batch(rows)
speedup = t_loop / max(t_batch, 1e-9)              # max(...) защищает от деления на ноль
print(f"\nЦикл: {t_loop:.2f}s, Батч: {t_batch:.2f}s")
print(f"Ускорение: x{speedup:.1f}   (> 5 ? {speedup > 5})")

# ПОЧЕМУ НЕ «В 10 000 РАЗ БЫСТРЕЕ»:
# executemany экономит только СЕТЕВЫЕ ОБРАЩЕНИЯ (пинг-понг клиент <-> сервер).
# Всё остальное от способа вставки не зависит: сервер всё равно разбирает и выполняет вставку
# каждой строки, обновляет индекс PRIMARY KEY, пишет WAL и делает commit.
# К тому же сервер на localhost (сетевая задержка почти нулевая), так что по настоящей сети
# выигрыш был бы больше. Для миллионов строк быстрее всего COPY (cur.copy()).

      N |  цикл, c | batch, c | ускорение
     10 |    0.008 |    0.005 | x1.6
    100 |    0.010 |    0.006 | x1.7
   1000 |    0.081 |    0.012 | x6.8
  10000 |    0.774 |    0.080 | x9.7

Цикл: 0.78s, Батч: 0.08s
Ускорение: x9.5   (> 5 ? True)


## 5. SQLAlchemy Core

**Core** — это слой выражений SQL поверх DB-API. Даёт:

- **Пул соединений** из коробки (`QueuePool`).
- **Параметризацию:** `text("WHERE x = :x")` + `[{...}, {...}]` — автоматически превращается в `executemany`.
- Единый интерфейс к PostgreSQL / SQLite / MySQL.

Ключевые вещи:
- `create_engine(url, echo=False)` — движок, управляет пулом. `echo=True` печатает SQL в консоль (обязательно при отладке ORM).
- `text("SELECT ... :param")` — обёртка над строкой SQL. Параметры — именованные `:name`. **f-строки внутрь `text()` нельзя.**
- `engine.connect()` → `conn.execute(stmt, params)` → `conn.commit()`.
- Если второй аргумент `execute` — список словарей, Core сам выполнит `executemany`.


In [11]:
# SQLAlchemy Core: Продвинутая библиотека, которая 
# сама управляет пулом соединений (не надо открывать/закрывать связь с базой каждый раз).
# Пул (Connection Pool) — это заранее созданный набор «готовых к работе» соединений с базой данных.

from sqlalchemy import create_engine, text

engine = create_engine(SQLALCHEMY_URL, echo=False)

# --- DDL + батчевая вставка через Core ---
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS core_demo"))
    conn.execute(text("CREATE TABLE core_demo (id INT PRIMARY KEY, sensor TEXT, val NUMERIC)"))
    conn.commit()

    records = [
        {"id": i, "s": f"SENSOR_{i % 7}", "v": round((i * 0.37) % 100, 2)}
        for i in range(1, 501)
    ]
    # список dict -> автоматически executemany
    conn.execute(
        text("INSERT INTO core_demo (id, sensor, val) VALUES (:id, :s, :v)"),
        records,
    )
    conn.commit()
    print(f"Вставлено {len(records)} строк через Core.executemany")

    # Простой агрегат через Core
    row = conn.execute(
        text("SELECT count(*), max(val) FROM core_demo WHERE val > :threshold"),
        {"threshold": 50},
    ).one()
    print(f"Строк с val > 50: {row[0]}, максимум: {row[1]}")


Вставлено 500 строк через Core.executemany
Строк с val > 50: 230, максимум: 99.9


### Упражнение 5. Core: создать таблицу и залить батч

**Задание.** Через `text()` и `engine.connect()` создайте таблицу `ex5_measurements (id, sensor, value)` и вставьте 200 строк списком словарей.

**Шаги.**
1. `DROP TABLE IF EXISTS ex5_measurements` и `CREATE TABLE ...`.
2. Сгенерируйте `records` — список словарей `{"id", "sensor", "value"}` на 200 элементов.
3. Один вызов `conn.execute(text("INSERT ..."), records)`.
4. Проверьте: `SELECT count(*) FROM ex5_measurements WHERE value > 50`.

**Критерий успеха.** В консоли: `count > 0`. В логах — один `INSERT` (executemany), не 200.


In [24]:
# УПРАЖНЕНИЕ 5
# SQLAlchemy Core: если вторым аргументом execute передать СПИСОК СЛОВАРЕЙ,
# Core сам выполнит executemany (одна пачка вместо 200 отдельных запросов).

from sqlalchemy import text

with engine.connect() as conn:
    # 1. DDL: пересоздаём таблицу
    conn.execute(text("DROP TABLE IF EXISTS ex5_measurements"))
    conn.execute(text("CREATE TABLE ex5_measurements (id INT PRIMARY KEY, sensor TEXT, value NUMERIC)"))
    conn.commit()

    # 2. 200 записей. Ключи словаря ("id","sensor","value") должны совпадать с именами :id, :sensor, :value
    records = [{"id": i, "sensor": f"S{i % 4}", "value": i * 0.7} for i in range(1, 201)]   # датчики S0..S3

    # 3. ОДИН вызов: список словарей -> executemany
    conn.execute(
        text("INSERT INTO ex5_measurements (id, sensor, value) VALUES (:id, :sensor, :value)"),
        records,
    )
    conn.commit()

    # 4. Проверки
    total = conn.execute(text("SELECT count(*) FROM ex5_measurements")).scalar()
    print("всего строк:", total)                                  # 200
    cnt = conn.execute(
        text("SELECT count(*) FROM ex5_measurements WHERE value > :t"),   # :t — именованный параметр
        {"t": 50},
    ).scalar()                                                     # .scalar() = первое значение первой строки
    print("count > 50:", cnt)                                      # 129 (0.7*i > 50 при i >= 72)

всего строк: 200
count > 50: 129


## 6. Pandas + `chunksize`

`pandas` — in-memory аналитика. **DataFrame** — это таблица в RAM.

- `pd.read_sql(query, engine, chunksize=500)` возвращает **итератор** по чанкам. Так вы не выгрузите 10 ГБ в 8 ГБ RAM.
- `np.where(cond, a, b)` — векторизованный `CASE WHEN`. Работает на C, а не в Python-цикле.
- `chunk[chunk['col'] > 0]` — фильтрация тоже векторизована.

**Разделение труда:** PostgreSQL — для OLTP и сложных `JOIN`. Pandas — для математических трансформаций, feature engineering, агрегатов «в памяти».


In [15]:
import pandas as pd
import numpy as np

# Создаем словарь для накопления итоговых счетчиков (начинаем с нулей)
total = {"HIGH": 0, "LOW": 0}

# Читаем таблицу кусками по 200 строк (chunksize), чтобы не переполнить оперативную память
for chunk in pd.read_sql("SELECT * FROM core_demo", engine, chunksize=200):
    
    # Создаем новую колонку 'level'. np.where работает как "если... то... иначе..."
    # Если значение в колонке 'val' > 50, ставим "HIGH", иначе "LOW"
    chunk["level"] = np.where(chunk["val"] > 50, "HIGH", "LOW")
    
    # Считаем, сколько раз в ЭТОМ КУСКЕ встретилось "HIGH" и сколько "LOW"
    vc = chunk["level"].value_counts()
    
    # Проходимся по нашему итоговому словарю (k будет по очереди "HIGH", потом "LOW")
    for k in total:
        # Берем число из текущего куска (vc.get) и прибавляем к общему итогу.
        # Если в куске нет слова "HIGH", get(k, 0) вернет 0, чтобы не было ошибки.
        total[k] += int(vc.get(k, 0))

# Выводим итоговый словарь с общим количеством HIGH и LOW
print("По всей таблице:", total)

# Выводим общую сумму строк (складываем HIGH и LOW)
print("Строк всего   :", sum(total.values()))

По всей таблице: {'HIGH': 230, 'LOW': 270}
Строк всего   : 500


### Упражнение 6. Агрегат по сенсорам

**Задание.** Читая `ex5_measurements` чанками по 50, посчитайте среднее `value` и количество замеров по каждому `sensor`.

**Шаги.**
1. Итерируйтесь `pd.read_sql(..., engine, chunksize=50)`.
2. Для каждого чанка считайте `groupby("sensor").agg(mean=("value","mean"), n=("value","size"))`.
3. Соберите результаты в словарь `{sensor: {"sum": ..., "n": ...}}` (агрегируйте среднее, пересчитывая взвешенно).
4. Напечатайте итоговую таблицу.

**Критерий успеха.** Сумма `n` по всем сенсорам == общее число строк в `ex5_measurements`.


In [26]:
# УПРАЖНЕНИЕ 6
# Читаем таблицу КУСКАМИ по 50 строк и копим промежуточные итоги.
# Ловушка: нельзя усреднять средние кусков (куски могут быть разного размера).
# Правильно копить СУММУ и КОЛИЧЕСТВО, а среднее считать в самом конце: сумма / количество.

import pandas as pd

acc = {}   # итог: {sensor: {"sum": накопленная сумма, "n": накопленное количество}}

# value::float8 — приводим NUMERIC к обычному числу на стороне SQL,
# чтобы pandas получил числовую колонку, а не объекты Decimal
sql = "SELECT sensor, value::float8 AS value FROM ex5_measurements ORDER BY id"

# chunksize=50 -> read_sql возвращает ИТЕРАТОР: в памяти одновременно не больше 50 строк
for chunk in pd.read_sql(sql, engine, chunksize=50):
    # Группируем КУСОК по сенсору: сумма и количество внутри этого куска
    part = chunk.groupby("sensor").agg(s=("value", "sum"), n=("value", "size"))
    for sensor, row in part.iterrows():
        d = acc.setdefault(sensor, {"sum": 0.0, "n": 0})   # если сенсора ещё нет — заводим нули
        d["sum"] += float(row["s"])                         # прибавляем сумму куска
        d["n"]   += int(row["n"])                           # прибавляем количество куска

# Итоговая таблица: (sensor, mean, count)
result = pd.DataFrame(
    [(s, a["sum"] / a["n"], a["n"]) for s, a in sorted(acc.items())],
    columns=["sensor", "mean", "n"],
)
print(result.to_string(index=False))

# Критерий успеха: сумма n по всем сенсорам == числу строк в таблице
with engine.connect() as conn:
    rows_in_table = conn.execute(text("SELECT count(*) FROM ex5_measurements")).scalar()
print("\nсумма n =", result["n"].sum(), "| строк в таблице =", rows_in_table,
      "| совпало:", result["n"].sum() == rows_in_table)       # 200 == 200 -> True

sensor  mean  n
    S0  71.4 50
    S1  69.3 50
    S2  70.0 50
    S3  70.7 50

сумма n = 200 | строк в таблице = 200 | совпало: True


## 7. SQLAlchemy ORM

ORM мапит таблицы на Python-классы. Удобно для CRUD, но требует дисциплины.
(_ORM (SQLAlchemy): Позволяет работать с базой не через SQL, а через Python-классы (таблицы = классы, строки = объекты)._)

- Наследуемся от `DeclarativeBase`; колонки описываем через `Mapped[T] = mapped_column(...)`.
- **M:N** — через ассоциативную `Table`, в `relationship(secondary=assoc)`.
- **Каскад** — `ondelete='CASCADE'` **на уровне БД** (в `ForeignKey`). Это быстрее, чем ORM-каскад: PG удалит сам, без выборки всех дочерних строк в память.
- `with Session(engine) as s:` — сессия. `s.add(obj)`, `s.commit()`, `s.delete(obj)`, `s.scalars(select(...))`.
- `engine.echo = True` — печатать SQL в консоль (включайте при отладке!).


In [17]:
from sqlalchemy import ForeignKey, String, Table, Column, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base(DeclarativeBase):
    pass

pilot_aircraft = Table(
    "pilot_aircraft", Base.metadata,
    Column("pilot_id", ForeignKey("pilots.id", ondelete="CASCADE"), primary_key=True),
    Column("aircraft_code", ForeignKey("aircrafts_orm.aircraft_code", ondelete="CASCADE"), primary_key=True),
)

class Aircraft(Base):
    __tablename__ = "aircrafts_orm"
    aircraft_code: Mapped[str] = mapped_column(String(3), primary_key=True)
    model:         Mapped[str] = mapped_column(String(50))
    pilots: Mapped[list["Pilot"]] = relationship(secondary=pilot_aircraft, back_populates="aircrafts")

class Pilot(Base):
    __tablename__ = "pilots"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    aircrafts: Mapped[list["Aircraft"]] = relationship(secondary=pilot_aircraft, back_populates="pilots")

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

with Session(engine) as s:
    boeing = Aircraft(aircraft_code="773", model="Boeing 777-300")
    p1 = Pilot(name="Иван", aircrafts=[boeing])
    s.add(p1)
    s.commit()

    # Каскадное удаление
    to_del = s.query(Pilot).filter_by(name="Иван").first()
    s.delete(to_del)
    s.commit()

    # Связей не осталось?
    links = s.execute(select(pilot_aircraft)).fetchall()
    print("Связей после каскада:", len(links))   # 0


Связей после каскада: 0


### Упражнение 7. Своя M:N-модель

**Задание.** Опишите M:N между `Author` и `Book` через ассоциативную `author_book`.

**Шаги.**
1. Объявите `Author` (id, name) и `Book` (id, title) и `author_book` через `Table`.
2. В `relationship` пропишите `secondary=author_book` и `back_populates` с обеих сторон.
3. `create_all`.
4. Добавьте 2 авторов и 3 книги (некоторые — с двумя авторами).
5. Удалите одного автора и проверьте, что связи в `author_book` для него исчезли, а книги, у которых были другие авторы, остались.

**Критерий успеха.** До удаления связей 3–4, после удаления — меньше на число связей удалённого автора.


In [27]:
# УПРАЖНЕНИЕ 7
# M:N («многие ко многим»): у автора много книг, у книги может быть несколько авторов.
# В БД это три таблицы: authors, books и связующая author_book.

from sqlalchemy import ForeignKey, String, Table, Column, select, delete, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

# 1. Свой базовый класс 
class Base2(DeclarativeBase):
    pass

# 2. Связующая таблица. ondelete="CASCADE" — правило ВНУТРИ БД:
#    удалили автора (или книгу) -> строки связи удаляются сами.
#    Составной первичный ключ (author_id, book_id) запрещает дубль пары «автор–книга».
author_book = Table(
    "author_book", Base2.metadata,
    Column("author_id", ForeignKey("authors.id", ondelete="CASCADE"), primary_key=True),
    Column("book_id",   ForeignKey("books.id",   ondelete="CASCADE"), primary_key=True),
)

# 3. Автор. secondary= через какую таблицу связаны; back_populates= имя «обратной» стороны
class Author(Base2):
    __tablename__ = "authors"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    books: Mapped[list["Book"]] = relationship(secondary=author_book, back_populates="authors")

# 4. Книга
class Book(Base2):
    __tablename__ = "books"
    id:    Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    authors: Mapped[list["Author"]] = relationship(secondary=author_book, back_populates="books")

# 5. Чистый старт: drop_all, чтобы при повторном запуске числа не «плыли»
Base2.metadata.drop_all(engine)
Base2.metadata.create_all(engine)

def count_links(s):
    """Сколько строк сейчас в связующей таблице."""
    return s.execute(select(func.count()).select_from(author_book)).scalar()

def count_books(s):
    return s.execute(select(func.count()).select_from(Book)).scalar()

with Session(engine) as s:
    # ВАЖНО: каждую книгу создаём ОДИН раз и отдаём ОДИН И ТОТ ЖЕ объект двум авторам.
    # Если написать Book(title="Война и мир") дважды, получатся две разные строки в books,
    # и общей книги не будет.
    war   = Book(title="Война и мир")      # общая книга
    anna  = Book(title="Анна Каренина")
    idiot = Book(title="Идиот")

    tolstoy = Author(name="Толстой",     books=[war, anna])     # 2 связи
    dosto   = Author(name="Достоевский", books=[war, idiot])    # 2 связи
    s.add_all([tolstoy, dosto])          # книги сохранятся автоматически через связи
    s.commit()

    print("книг:", count_books(s), "| связей ДО удаления:", count_links(s))   # 3 и 4

    # 6. Удаляем Толстого
    s.delete(tolstoy)
    s.commit()

    print("связей ПОСЛЕ удаления Толстого:", count_links(s))   # 2 (4 минус его 2 связи)
    print("книг осталось:", count_books(s))                    # 3 — сами книги НЕ удаляются

    # Проверяем общую книгу: у «Войны и мира» остался один автор — Достоевский
    war_authors = s.scalars(
        select(Author.name).join(author_book).join(Book).where(Book.title == "Война и мир")
    ).all()
    print("авторы «Войны и мира» теперь:", war_authors)         # ['Достоевский']

    # БОНУС: «массовое» удаление мимо ORM — здесь работает именно CASCADE на уровне БД
    s.execute(delete(Author).where(Author.name == "Достоевский"))
    s.commit()
    print("связей после массового удаления:", count_links(s))  # 0

# Заметка: при обычном s.delete(obj) строки связующей таблицы убирает сам ORM.
# ondelete="CASCADE" страхует случаи, когда удаляют напрямую SQL-ом (как в бонусе).

книг: 3 | связей ДО удаления: 4
связей ПОСЛЕ удаления Толстого: 2
книг осталось: 3
авторы «Войны и мира» теперь: ['Достоевский']
связей после массового удаления: 0


## 8. Проблема N+1 и `selectinload`

**N+1** — это когда вы делаете **1** запрос за родителями, а потом в цикле дёргаете `p.children` — и ORM делает ещё **N** запросов (по одному на родителя). Итого `N+1` сетевых обращений вместо двух.

Как поймать: `engine.echo=True` — в логах видно один `SELECT ... FROM parents` и N однотипных `SELECT ... FROM children WHERE parent_id = ...`.

Как лечить:
- `selectinload(Parent.children)` — ровно **2** запроса: `SELECT parents` + `SELECT children WHERE parent_id IN (...)`. Часто рекомендуемый дефолт.
- `joinedload(Parent.children)` — **1** запрос с `LEFT OUTER JOIN`, но с дублями родительских колонок. На широких таблицах — лишний трафик. Требует `.unique()` при использовании с `scalars().all()`.


In [20]:
from sqlalchemy import event
from sqlalchemy.orm import selectinload

def count_queries(fn):
    """Считает SQL-запросы, которые выполнит fn()."""
    counter = {"n": 0}
    def before(conn, cursor, statement, params, ctx, many):
        counter["n"] += 1
    event.listen(engine, "before_cursor_execute", before)
    try:
        result = fn()
    finally:
        event.remove(engine, "before_cursor_execute", before)
    return result, counter["n"]

# Готовим данные: 5 пилотов, у каждого 2 самолёта
with Session(engine) as s:
    s.query(Pilot).delete()
    s.query(Aircraft).delete()
    s.commit()
    for i in range(5):
        a1 = Aircraft(aircraft_code=f"A{i}1", model=f"Model {i}-1")
        a2 = Aircraft(aircraft_code=f"A{i}2", model=f"Model {i}-2")
        s.add(Pilot(name=f"Pilot_{i}", aircrafts=[a1, a2]))
    s.commit()

def lazy_load():
    with Session(engine) as s:
        pilots = s.scalars(select(Pilot)).all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

def eager_load():
    with Session(engine) as s:
        stmt = select(Pilot).options(selectinload(Pilot.aircrafts))
        pilots = s.scalars(stmt).all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

_, n_lazy  = count_queries(lazy_load)
_, n_eager = count_queries(eager_load)
print(f"ленивая (N+1)     : {n_lazy} запросов")
print(f"selectinload      : {n_eager} запросов")


ленивая (N+1)     : 6 запросов
selectinload      : 2 запросов


### Упражнение 8. `joinedload` и `.unique()`

**Задание.** Решите ту же задачу через `joinedload` и убедитесь, что запрос действительно один.

**Шаги.**
1. Импортируйте `joinedload` из `sqlalchemy.orm`.
2. Замените `selectinload(Pilot.aircrafts)` на `joinedload(Pilot.aircrafts)`.
3. Запустите `eager_load()` — получите `InvalidRequestError` (нужна дедупликация).
4. Исправьте: `s.scalars(stmt).unique().all()`.
5. Пересчитайте запросы через `count_queries`.

**Критерий успеха.** 1 запрос вместо 2. В комментарии — почему `selectinload` всё равно рекомендуют чаще.


In [28]:
# УПРАЖНЕНИЕ 8
# N+1: 1 запрос за родителями + N запросов за детьми каждого.
# Лечим: selectinload (2 запроса) или joinedload (1 запрос с JOIN, но нужен .unique()).

from sqlalchemy.orm import joinedload
from sqlalchemy.exc import InvalidRequestError

# --- Шаг 3 задания: joinedload БЕЗ .unique() -> ошибка ---
def eager_load_joined_wrong():
    with Session(engine) as s:
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        return len(s.scalars(stmt).all())          # здесь будет InvalidRequestError

try:
    count_queries(eager_load_joined_wrong)
except InvalidRequestError as e:
    # JOIN размножает пилота по числу его самолётов (2 строки на пилота),
    # и SQLAlchemy требует явно «схлопнуть» дубли
    print("Без unique():", str(e).splitlines()[0][:100], "...")

# --- Шаг 4: исправленная версия ---
def eager_load_joined():
    with Session(engine) as s:
        # 1. Один запрос с LEFT OUTER JOIN: пилоты и их самолёты приходят вместе
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        # 2. .unique() склеивает дубли пилотов обратно в одного
        pilots = s.scalars(stmt).unique().all()
        for p in pilots:
            _ = p.aircrafts                         # данные уже в памяти -> новых запросов нет
        return len(pilots)

# --- Шаг 5: считаем запросы всеми тремя способами ---
_, n_joined = count_queries(eager_load_joined)
print("ленивая (N+1)  :", n_lazy,   "запросов")    # 6 = 1 + 5
print("selectinload   :", n_eager,  "запроса")     # 2
print("joinedload     :", n_joined, "запрос")      # 1

# ПОЧЕМУ selectinload РЕКОМЕНДУЮТ ЧАЩЕ:
# 1) joinedload повторяет ВСЕ колонки родителя в каждой строке JOIN -> лишний трафик на «широких» таблицах.
# 2) При нескольких коллекциях сразу строки перемножаются (m x n), результат раздувается.
# 3) С LIMIT по родителям joinedload приходится оборачивать в подзапрос, запрос усложняется.
# 4) selectinload шлёт простой второй запрос "... WHERE parent_id IN (...)", данные не дублируются.
# joinedload хорош для связей «многие-к-одному» и когда критично именно число запросов.

Без unique(): The unique() method must be invoked on this Result, as it contains results that include joined eager ...
ленивая (N+1)  : 6 запросов
selectinload   : 2 запроса
joinedload     : 1 запрос


## Итоги семинара

1. **psycopg:** соединения через `with`, курсоры через `with`, всегда параметризация `%s`.
2. **Транзакции:** `with conn:` → commit/rollback; короткие транзакции; DDL транзакционен.
3. **SQL-инъекции:** f-string в SQL — уязвимо; `%s` и `:name` — безопасно.
4. **executemany:** один вызов вместо N `execute`; для миллионов строк — `COPY`.
5. **SQLAlchemy Core:** `text()` + `engine.connect()` + `executemany` из списка словарей.
6. **Pandas:** `read_sql(chunksize=…)` и `np.where` — векторизация; PostgreSQL для JOIN, pandas — для фич.
7. **ORM:** `DeclarativeBase`, `relationship(secondary=…)`, `ondelete='CASCADE'` на уровне БД.
8. **N+1:** ловится `echo=True`, лечится `selectinload` (обычно) или `joinedload().unique()` (когда важна экономия на количестве запросов).


---

# Решения

> Дальше — эталонные решения всех 8 упражнений. **Не подглядывайте, пока не попробовали.**


In [ ]:
# РЕШЕНИЕ 1
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", ("apple",))
        print("apple :", cur.fetchall())
        cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s", ("apple' OR '1'='1",))
        print("inject:", cur.fetchall())  # []


In [ ]:
# РЕШЕНИЕ 2
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS ex2")
        cur.execute("CREATE TABLE ex2 (id INT PRIMARY KEY, name TEXT)")
    conn.commit()

try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.executemany("INSERT INTO ex2 VALUES (%s, %s)", [(1, "a"), (2, "b"), (1, "dup")])
        conn.commit()
except UniqueViolation as e:
    print("caught:", str(e).splitlines()[0])

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM ex2")
        print("rows after rollback:", cur.fetchone()[0])  # 0


In [ ]:
# РЕШЕНИЕ 3
def fixed(code):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT id, name, qty FROM demo_items WHERE name = %s LIMIT 5", (code,))
            return cur.fetchall()

print("broken:", broken("' OR '1'='1"))
print("fixed :", fixed("' OR '1'='1"))


In [ ]:
# РЕШЕНИЕ 4
import time, random

N = 10_000
rows = [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, N + 1)]

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench10k")
        cur.execute("CREATE TABLE bench10k (id INT PRIMARY KEY, name TEXT, val INT)")
    conn.commit()

t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        for r in rows:
            cur.execute("INSERT INTO bench10k VALUES (%s, %s, %s)", r)
    conn.commit()
t_loop = time.perf_counter() - t0

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE bench10k")
    conn.commit()

t0 = time.perf_counter()
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO bench10k VALUES (%s, %s, %s)", rows)
    conn.commit()
t_batch = time.perf_counter() - t0

print(f"loop={t_loop:.2f}s  batch={t_batch:.2f}s  speedup=x{t_loop/max(t_batch,1e-9):.1f}")
# Почему не «в N раз быстрее» линейно: часть времени уходит на commit/WAL,
# а с ростом батча растёт время на один пакет; executemany уже почти линейно-эффективен.


In [ ]:
# РЕШЕНИЕ 5
with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS ex5_measurements"))
    conn.execute(text("CREATE TABLE ex5_measurements (id INT PRIMARY KEY, sensor TEXT, value NUMERIC)"))
    conn.commit()
    records = [{"id": i, "sensor": f"S{i%4}", "value": i * 0.7} for i in range(1, 201)]
    conn.execute(
        text("INSERT INTO ex5_measurements (id, sensor, value) VALUES (:id, :sensor, :value)"),
        records,
    )
    conn.commit()
    cnt = conn.execute(text("SELECT count(*) FROM ex5_measurements WHERE value > :t"), {"t": 50}).scalar()
    print("count > 50:", cnt)


In [ ]:
# РЕШЕНИЕ 6
import pandas as pd

agg = {}
for chunk in pd.read_sql("SELECT sensor, value FROM ex5_measurements", engine, chunksize=50):
    g = chunk.groupby("sensor").agg(mean=("value", "mean"), n=("value", "size"))
    for sensor, row in g.iterrows():
        if sensor not in agg:
            agg[sensor] = {"sum": 0.0, "n": 0}
        agg[sensor]["sum"] += float(row["mean"]) * int(row["n"])
        agg[sensor]["n"]   += int(row["n"])

for s, a in sorted(agg.items()):
    print(f"{s}: mean={a['sum']/a['n']:.2f}, n={a['n']}")
print("total n:", sum(a["n"] for a in agg.values()))


In [ ]:
# РЕШЕНИЕ 7
from sqlalchemy import ForeignKey, String, Table, Column, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base2(DeclarativeBase):
    pass

author_book = Table(
    "author_book", Base2.metadata,
    Column("author_id", ForeignKey("authors.id", ondelete="CASCADE"), primary_key=True),
    Column("book_id",   ForeignKey("books.id",   ondelete="CASCADE"), primary_key=True),
)

class Author(Base2):
    __tablename__ = "authors"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    books: Mapped[list["Book"]] = relationship(secondary=author_book, back_populates="authors")

class Book(Base2):
    __tablename__ = "books"
    id:    Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    authors: Mapped[list["Author"]] = relationship(secondary=author_book, back_populates="books")

Base2.metadata.drop_all(engine)
Base2.metadata.create_all(engine)

with Session(engine) as s:
    tolstoy = Author(name="Толстой", books=[Book(title="Война и мир"), Book(title="Анна Каренина")])
    dosto   = Author(name="Достоевский", books=[Book(title="Идиот"), Book(title="Война и мир")])
    s.add_all([tolstoy, dosto])
    s.commit()
    print("links before:", len(s.execute(select(author_book)).fetchall()))

    s.delete(s.query(Author).filter_by(name="Толстой").first())
    s.commit()
    print("links after :", len(s.execute(select(author_book)).fetchall()))
    print("books left  :", s.query(Book).count())


In [ ]:
# РЕШЕНИЕ 8
from sqlalchemy.orm import joinedload

def eager_load_joined():
    with Session(engine) as s:
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        pilots = s.scalars(stmt).unique().all()
        for p in pilots:
            _ = p.aircrafts
        return len(pilots)

_, n_joined = count_queries(eager_load_joined)
print("joinedload queries:", n_joined)  # 1
# Комментарий: selectinload предпочтительнее для больших коллекций,
# потому что не дублирует родительские колонки в каждой строке JOIN.
